# Naive Bayes:

---

# Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

## Importing Dataset:

In [2]:
data = pd.read_csv("email.csv")

In [3]:
data.drop_duplicates(keep='first', inplace=True)

## Feature Engineering:

In [4]:
import re
import html

EMAIL_PATTERN = re.compile(r'\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b')

URL_PATTERN = re.compile(
    r'(?:https?://|www\.)\S+'
    r'|\b[a-z0-9\-]+(?:\.[a-z0-9\-]+)*\.(?:com|net|org|info|biz|tv|co\.uk|uk)\b(?:/\S*)?',
    flags=re.IGNORECASE,
)

PHONE_PATTERN = re.compile(r'(?<!\d)\+?\d(?:[\s\-]?\d){9,13}(?!\d)')


def clean_text(text):
    text = html.unescape(str(text))                              # &amp; -> &, &lt;#&gt; -> <#>
    text = text.lower().strip()                                  # lowercase + strip whitespace
    text = EMAIL_PATTERN.sub(' email ', text)                    # emails first, so they don't become half-URLs
    text = URL_PATTERN.sub(' url ', text)                        # links
    text = PHONE_PATTERN.sub(' phone-number ', text)             # phone numbers
    return re.sub(r'\s+', ' ', text).strip()                     # collapse repeated spaces

In [5]:
data['Message'] = data['Message'].apply(clean_text)

## Train-Test-Split:

In [6]:
X = data.drop(columns=['Category'])
y = data['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

## Preprocessing:

In [7]:
# Count Vectorizer:

cv = CountVectorizer(stop_words='english')

X_train_cv = cv.fit_transform(X_train['Message'])
X_test_cv = cv.transform(X_test['Message'])

In [8]:
# TF-IDF Vectorizer:

tfidf = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train['Message'])
X_test_tfidf = tfidf.transform(X_test['Message'])

In [9]:
label = LabelEncoder()

y_train_transform = label.fit_transform(y_train)
y_test_transform = label.transform(y_test)

## Exploratory Data Analysis:

In [10]:
X_df = pd.DataFrame(np.concat((X_train_cv.toarray(), X_test_cv.toarray()), axis=0), columns=cv.get_feature_names_out(), index=data.index)

y_df = pd.DataFrame(np.concat((y_train_transform, y_test_transform), axis=0), columns=['Category'], index=data.index)

df = pd.concat([X_df, y_df], axis=1)

In [11]:
spam_words = df[df['Category'] == 1].drop(columns=['Category']).sum()
ham_words = df[df['Category'] == 0].drop(columns=['Category']).sum()

words_count = pd.DataFrame({'Spam Words' : spam_words,
                           'Ham Words' : ham_words})

top_spam = (words_count['Spam Words'].sort_values(ascending=False).head(10))
top_ham = (words_count['Ham Words'].sort_values(ascending=False).head(10))

print(f"Spam Words:\n{top_spam.to_dict()}")
print("\n")
print(f"Ham Words:\n{top_ham.to_dict()}")

Spam Words:
{'phone': 438, 'number': 422, 'free': 190, 'txt': 131, 'ur': 119, 'url': 118, 'stop': 110, 'text': 107, 'mobile': 105, 'reply': 96}


Ham Words:
{'just': 282, 'ok': 256, 'll': 233, 'got': 227, 'know': 226, 'like': 224, 'good': 217, 'come': 214, 'ur': 203, 'time': 193}


## Building Model:

### 1. Count Vectroizer:

In [12]:
nb = MultinomialNB()

start_time = time.time()
nb.fit(X_train_cv, y_train_transform)
end_time = time.time() - start_time

print("Model Trained Successfully!")
print(f"Time Taken for Training: {end_time:.2f}s")

Model Trained Successfully!
Time Taken for Training: 0.01s


In [13]:
y_pred_cv = nb.predict(X_test_cv)

print(f"Accuracy: {(accuracy_score(y_test_transform, y_pred_cv) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test_transform, y_pred_cv)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test_transform, y_pred_cv)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test_transform, y_pred_cv)*100):.2f}%")

Accuracy: 97.77%

Precision: 88.64%

Recall: 93.60%

F1 Score: 91.05%


In [14]:
matrix = confusion_matrix(y_test_transform, y_pred_cv)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,892,15
Actual spam,8,117


In [15]:
# without stop_words:

preprocessing = ColumnTransformer(transformers=[
    ('cv', CountVectorizer(stop_words='english'), 'Message')
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', MultinomialNB())
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.14%

Precision: 92.15%

Recall: 92.98%

F1: 92.55%



In [16]:
# with stop_words:

preprocessing = ColumnTransformer(transformers=[
    ('cv', CountVectorizer(), 'Message')
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', MultinomialNB())
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.47%

Precision: 96.40%

Recall: 91.11%

F1: 93.65%



### 2. TF-IDF Vectroizer:

In [17]:
nb = MultinomialNB()

start_time = time.time()
nb.fit(X_train_tfidf, y_train_transform)
end_time = time.time() - start_time

print("Model Trained Successfully!")
print(f"Time Taken for Training: {end_time:.2f}s")

Model Trained Successfully!
Time Taken for Training: 0.01s


In [18]:
y_pred_tfidf = nb.predict(X_test_tfidf)

print(f"Accuracy: {(accuracy_score(y_test_transform, y_pred_tfidf) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test_transform, y_pred_tfidf)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test_transform, y_pred_tfidf)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test_transform, y_pred_tfidf)*100):.2f}%")

Accuracy: 96.90%

Precision: 97.94%

Recall: 76.00%

F1 Score: 85.59%


In [19]:
matrix = confusion_matrix(y_test_transform, y_pred_tfidf)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,905,2
Actual spam,30,95


In [20]:
# without stop_words:

preprocessing = ColumnTransformer(transformers=[
    ('tfidf', TfidfVectorizer(stop_words='english'), 'Message')
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', MultinomialNB())
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 96.74%

Precision: 99.60%

Recall: 74.10%

F1: 84.95%



In [21]:
# with stop_words:

preprocessing = ColumnTransformer(transformers=[
    ('tfidf', TfidfVectorizer(), 'Message')
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', MultinomialNB())
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=model,
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 95.64%

Precision: 99.78%

Recall: 65.06%

F1: 78.73%

